### Cell 1 Configuration

In [ ]:
import os
import torch

class Config:
    CSV_TRAIN = "train.csv"
    IMG_DIR_TRAIN = "data/train"

    NUM_CLASSES = 3
    
    # Model Settings
    IMAGE_SIZE = 224
    BATCH_SIZE = 32
    NUM_WORKERS = 4  
    
    # Hyperparameters
    EPOCHS = 5
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-2
    VAL_SPLIT_RATIO = 0.2
    RANDOM_SEED = 42
    
    # Hardware Device Detection
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using Device: {Config.DEVICE}")
if Config.DEVICE.type == 'cuda':
    print(f"🔥 GPU Model: {torch.cuda.get_device_name(0)}")

Using Device: cuda
🔥 GPU Model: NVIDIA GeForce RTX 3050 6GB Laptop GPU


### Create the K-Fold object

In [73]:
from sklearn.model_selection import StratifiedKFold
NUM_FOLDS = 5

skf = StratifiedKFold(
    n_splits=NUM_FOLDS,
    shuffle=True,
    random_state=Config.RANDOM_SEED
)

### Dataset and DataLoader

In [74]:
# CELL 2: DATASET & DATALOADERS (Using ImageFolder)

import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from dataset import IlluminationDataset, train_transforms, val_transforms
from sklearn.model_selection import StratifiedKFold

In [75]:
# 1. Load the complete training CSV
full_df = pd.read_csv(Config.CSV_TRAIN)

fold_results = []

# 2. Create 5-Fold splits
for fold, (train_idx, val_idx) in enumerate(
    skf.split(full_df, full_df["label"]), start=1
):
    print("=" * 50)
    print(f"Fold {fold}/{NUM_FOLDS}")
    print("=" * 50)

    # Split the dataframe
    train_df = full_df.iloc[train_idx].reset_index(drop=True)
    val_df = full_df.iloc[val_idx].reset_index(drop=True)

    # Create datasets
    train_dataset = IlluminationDataset(
        dataframe=train_df,
        img_dir=Config.IMG_DIR_TRAIN,
        transform=train_transforms
    )

    val_dataset = IlluminationDataset(
        dataframe=val_df,
        img_dir=Config.IMG_DIR_TRAIN,
        transform=val_transforms
    )

    # Create DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=Config.BATCH_SIZE,
        shuffle=True,
        num_workers=Config.NUM_WORKERS,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=Config.BATCH_SIZE,
        shuffle=False,
        num_workers=Config.NUM_WORKERS,
        pin_memory=True
    )

    # Verify current fold
    print(f"Training images: {len(train_dataset)}")
    print(f"Validation images: {len(val_dataset)}")

    print("\nTraining class distribution:")
    print(train_df["label"].value_counts().sort_index())

    print("\nValidation class distribution:")
    print(val_df["label"].value_counts().sort_index())

Fold 1/5
Training images: 1200
Validation images: 300

Training class distribution:
label
0    400
1    400
2    400
Name: count, dtype: int64

Validation class distribution:
label
0    100
1    100
2    100
Name: count, dtype: int64
Fold 2/5
Training images: 1200
Validation images: 300

Training class distribution:
label
0    400
1    400
2    400
Name: count, dtype: int64

Validation class distribution:
label
0    100
1    100
2    100
Name: count, dtype: int64
Fold 3/5
Training images: 1200
Validation images: 300

Training class distribution:
label
0    400
1    400
2    400
Name: count, dtype: int64

Validation class distribution:
label
0    100
1    100
2    100
Name: count, dtype: int64
Fold 4/5
Training images: 1200
Validation images: 300

Training class distribution:
label
0    400
1    400
2    400
Name: count, dtype: int64

Validation class distribution:
label
0    100
1    100
2    100
Name: count, dtype: int64
Fold 5/5
Training images: 1200
Validation images: 300

Training 

### CELL 3: MODEL ARCHITECTURE & LAYER FREEZING

In [76]:
# CELL 3: MODEL ARCHITECTURE & LAYER FREEZING

import torch.nn as nn
import torchvision.models as models

def build_illumination_model():
    # Initialize a fresh model for every fold
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)

    # Freeze all layers
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze Layer4
    for param in model.layer4.parameters():
        param.requires_grad = True

    # Replace FC head
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(num_features, Config.NUM_CLASSES)
    )
    model = model.to(Config.DEVICE)


    return model


model = build_illumination_model().to(Config.DEVICE)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel() for p in model.parameters()
)

print(f"Model Initialized on {Config.DEVICE}")
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Model Initialized on cuda
Total Parameters: 11,178,051
Trainable Parameters: 8,395,267


#### Training on 8 million parameters

### CELL 4: TRAINING & VALIDATION ENGINE

In [77]:
from tqdm import tqdm # Great for visual progress bars in VS Code

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct_preds = 0
    total_samples = 0
    
    # Progress bar for the terminal/notebook
    loop = tqdm(dataloader, leave=False, desc="Training")
    
    for images, labels in loop:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        
        # 1. Forward Pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # 2. Backward Pass & Optimization
        
        loss.backward()
        optimizer.step()
        
        # 3. Track Metrics
        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, dim=1) # Get the index of the highest logit
        correct_preds += torch.sum(preds == labels).item()
        total_samples += labels.size(0)
        
        # Update progress bar
        loop.set_postfix(loss=loss.item())
        
    epoch_loss = running_loss / total_samples
    epoch_acc = correct_preds / total_samples
    return epoch_loss, epoch_acc

def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct_preds = 0
    total_samples = 0
    
    with torch.no_grad(): # Disable gradient tracking for speed and memory
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

             # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Track metrics
            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)
            correct_preds += torch.sum(preds == labels).item()
            total_samples += labels.size(0)
            
    epoch_loss = running_loss / total_samples
    epoch_acc = correct_preds / total_samples
    return epoch_loss, epoch_acc

In [78]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels[:10])

torch.Size([32, 3, 224, 224])
tensor([1, 0, 0, 0, 2, 0, 0, 1, 2, 0])


## TTA Transform


In [79]:
import torchvision.transforms as transforms

tta_transforms = [

    transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485,0.456,0.406],
            std=[0.229,0.224,0.225]
        )
    ]),

    transforms.Compose([
        transforms.Resize((224,224)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485,0.456,0.406],
            std=[0.229,0.224,0.225]
        )
    ]),

    transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ColorJitter(brightness=0.15),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485,0.456,0.406],
            std=[0.229,0.224,0.225]
        )
    ]),

    transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ColorJitter(contrast=0.15),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485,0.456,0.406],
            std=[0.229,0.224,0.225]
        )
    ])
]

### implement proper TTA. We'll average the softmax probabilities, not the predicted classes. This is the standard approach used in most computer vision competitions.

### Cell 1 — TTA Prediction Function

In [80]:
from PIL import Image
import torch
import torch.nn.functional as F

def predict_with_tta(model, image_path, tta_transforms, device):
    """
    Predict a single image using Test-Time Augmentation (TTA).
    Returns:
        avg_probs : Averaged class probabilities
        pred_class: Final predicted class
    """

    model.eval()

    image = Image.open(image_path).convert("RGB")

    probs = []

    with torch.no_grad():

        for transform in tta_transforms:

            img = transform(image).unsqueeze(0).to(device)

            output = model(img)

            prob = F.softmax(output, dim=1)

            probs.append(prob)

        # Average probabilities
        avg_probs = torch.mean(torch.stack(probs), dim=0)

        pred_class = torch.argmax(avg_probs, dim=1).item()

    return avg_probs.cpu(), pred_class

### Cell 2 — Class Mapping

In [81]:
idx_to_class = {
    0: "dark",
    1: "normal",
    2: "bright"
}

### Cell 3 — Test on One Image

In [82]:
image_path = "data/train/bright/0b8ebc21-dde0-4c94-acc0-7522b548e7fa.png"

probs, pred = predict_with_tta(
    model,
    image_path,
    tta_transforms,
    Config.DEVICE
)

print("Predicted Class :", idx_to_class[pred])
print("Probabilities   :", probs.numpy())

Predicted Class : bright
Probabilities   : [[0.21000178 0.3330927  0.45690554]]


### CELL 5: MAIN EXECUTION LOOP

In [ ]:
import time
criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

# Optimizer
optimizer = torch.optim.AdamW(
    [
        {
            "params": model.layer4.parameters(),
            "lr": 5e-5
        },
        {
            "params": model.fc.parameters(),
            "lr": 2e-4
        }
    ],
    weight_decay=Config.WEIGHT_DECAY
)

# Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

print(f"Model initialized for Fold {fold}")
# 2. Initialize Tracking Variables
best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print("🔥 Starting Training...")
start_time = time.time()

for epoch in range(Config.EPOCHS):

        print(f"\nEpoch {epoch+1}/{Config.EPOCHS}")
        print("-" * 20)

        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            Config.DEVICE
        )

        val_loss, val_acc = validate_one_epoch(
            model,
            val_loader,
            criterion,
            Config.DEVICE
        )

        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        backbone_lr = optimizer.param_groups[0]["lr"]
        fc_lr = optimizer.param_groups[1]["lr"]

        print(f"Backbone LR : {backbone_lr:.8f}")
        print(f"FC LR       : {fc_lr:.8f}")

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss  : {val_loss:.4f} | Val Acc  : {val_acc:.4f}")

        if val_acc > best_val_acc:

            best_val_acc = val_acc

            torch.save(
                model.state_dict(),
                f"best_model_fold_{fold}.pth"
            )

            print(f"⭐ Best Model Saved (Fold {fold})")

    # ---------------------------------
    # Fold Summary
    # ---------------------------------
        time_elapsed = time.time() - start_time

        print("\n" + "-" * 50)
        
        print(f"Best Validation Accuracy : {best_val_acc:.4f}")
        print(
            f"Training Time : "
            f"{time_elapsed//60:.0f}m "
            f"{time_elapsed%60:.0f}s"
        )
        print("-" * 50)
print(f"Fold {fold} Completed")

fold_results.append(best_val_acc)

# ===========================================
# Final Cross Validation Result
# ===========================================

print("\n" + "=" * 60)
print("5-FOLD CROSS VALIDATION RESULTS")
print("=" * 60)

for i, acc in enumerate(fold_results, start=1):
    print(f"Fold {i}: {acc:.4f}")

print(f"\nAverage Validation Accuracy: {sum(fold_results)/len(fold_results):.4f}")
print(f"Best Fold Accuracy: {max(fold_results):.4f}")

Model initialized for Fold 5
🔥 Starting Training...

Epoch 1/15
--------------------


Backbone LR : 0.00005000
FC LR       : 0.00020000
Train Loss: 1.1770 | Train Acc: 0.3892
Val Loss  : 1.0796 | Val Acc  : 0.4500
⭐ Best Model Saved (Fold 5)

--------------------------------------------------
Fold 5 Completed
Best Validation Accuracy : 0.4500
Training Time : 0m 10s
--------------------------------------------------

Epoch 2/15
--------------------


Backbone LR : 0.00005000
FC LR       : 0.00020000
Train Loss: 1.0493 | Train Acc: 0.4758
Val Loss  : 1.0663 | Val Acc  : 0.4733
⭐ Best Model Saved (Fold 5)

--------------------------------------------------
Fold 5 Completed
Best Validation Accuracy : 0.4733
Training Time : 0m 20s
--------------------------------------------------

Epoch 3/15
--------------------


Backbone LR : 0.00005000
FC LR       : 0.00020000
Train Loss: 0.9688 | Train Acc: 0.5633
Val Loss  : 1.0423 | Val Acc  : 0.4900
⭐ Best Model Saved (Fold 5)

--------------------------------------------------
Fold 5 Completed
Best Validation Accuracy : 0.4900
Training Time : 0m 29s
--------------------------------------------------

Epoch 4/15
--------------------


Backbone LR : 0.00005000
FC LR       : 0.00020000
Train Loss: 0.9171 | Train Acc: 0.5825
Val Loss  : 1.0580 | Val Acc  : 0.4700

--------------------------------------------------
Fold 5 Completed
Best Validation Accuracy : 0.4900
Training Time : 0m 39s
--------------------------------------------------

Epoch 5/15
--------------------


...

Backbone LR : 0.00001250
FC LR       : 0.00005000
Train Loss: 0.5885 | Train Acc: 0.8517
Val Loss  : 1.1273 | Val Acc  : 0.4933

--------------------------------------------------
Fold 5 Completed
Best Validation Accuracy : 0.5067
Training Time : 1m 46s
--------------------------------------------------

Epoch 12/15
--------------------


Backbone LR : 0.00000625
FC LR       : 0.00002500
Train Loss: 0.5603 | Train Acc: 0.8658
Val Loss  : 1.1367 | Val Acc  : 0.4733

--------------------------------------------------
Fold 5 Completed
Best Validation Accuracy : 0.5067
Training Time : 1m 57s
--------------------------------------------------

Epoch 13/15
--------------------


Backbone LR : 0.00000625
FC LR       : 0.00002500
Train Loss: 0.5563 | Train Acc: 0.8833
Val Loss  : 1.1418 | Val Acc  : 0.4800

--------------------------------------------------
Fold 5 Completed
Best Validation Accuracy : 0.5067
Training Time : 2m 7s
--------------------------------------------------

Epoch 14/15
--------------------


Backbone LR : 0.00000625
FC LR       : 0.00002500
Train Loss: 0.5480 | Train Acc: 0.8908
Val Loss  : 1.1459 | Val Acc  : 0.4800

--------------------------------------------------
Fold 5 Completed
Best Validation Accuracy : 0.5067
Training Time : 2m 18s
--------------------------------------------------

Epoch 15/15
--------------------


Backbone LR : 0.00000313
FC LR       : 0.00001250
Train Loss: 0.5250 | Train Acc: 0.9025
Val Loss  : 1.1520 | Val Acc  : 0.4900

--------------------------------------------------
Fold 5 Completed
Best Validation Accuracy : 0.5067
Training Time : 2m 29s
--------------------------------------------------

5-FOLD CROSS VALIDATION RESULTS
Fold 1: 0.5067

Average Validation Accuracy: 0.5067
Best Fold Accuracy: 0.5067


In [84]:
for epoch in range(len(history['train_loss'])):
    print(
        f"Epoch {epoch+1:02d} | "
        f"Train Acc: {history['train_acc'][epoch]:.4f} | "
        f"Val Acc: {history['val_acc'][epoch]:.4f} | "
        f"Train Loss: {history['train_loss'][epoch]:.4f} | "
        f"Val Loss: {history['val_loss'][epoch]:.4f}"
    )

Epoch 01 | Train Acc: 0.3892 | Val Acc: 0.4500 | Train Loss: 1.1770 | Val Loss: 1.0796
Epoch 02 | Train Acc: 0.4758 | Val Acc: 0.4733 | Train Loss: 1.0493 | Val Loss: 1.0663
Epoch 03 | Train Acc: 0.5633 | Val Acc: 0.4900 | Train Loss: 0.9688 | Val Loss: 1.0423
Epoch 04 | Train Acc: 0.5825 | Val Acc: 0.4700 | Train Loss: 0.9171 | Val Loss: 1.0580
Epoch 05 | Train Acc: 0.6550 | Val Acc: 0.4833 | Train Loss: 0.8269 | Val Loss: 1.0629
Epoch 06 | Train Acc: 0.7250 | Val Acc: 0.4533 | Train Loss: 0.7517 | Val Loss: 1.0938
Epoch 07 | Train Acc: 0.7633 | Val Acc: 0.4667 | Train Loss: 0.6941 | Val Loss: 1.0969
Epoch 08 | Train Acc: 0.8100 | Val Acc: 0.5067 | Train Loss: 0.6452 | Val Loss: 1.1015
Epoch 09 | Train Acc: 0.8100 | Val Acc: 0.4800 | Train Loss: 0.6398 | Val Loss: 1.1054
Epoch 10 | Train Acc: 0.8525 | Val Acc: 0.4900 | Train Loss: 0.5982 | Val Loss: 1.1166
Epoch 11 | Train Acc: 0.8517 | Val Acc: 0.4933 | Train Loss: 0.5885 | Val Loss: 1.1273
Epoch 12 | Train Acc: 0.8658 | Val Acc: 0.4

In [85]:
model.load_state_dict(
    torch.load(
        "best_model.pth",
        map_location=Config.DEVICE,
        weights_only=True
    )
)

model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [86]:
from sklearn.metrics import confusion_matrix, classification_report

all_labels = []
all_preds = []

model.eval()

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(Config.DEVICE)
        labels = labels.to(Config.DEVICE)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_preds,
        target_names=["dark", "normal", "bright"]
    )
)

Confusion Matrix:
[[82  6 12]
 [12 73 15]
 [19 14 67]]

Classification Report:
              precision    recall  f1-score   support

        dark       0.73      0.82      0.77       100
      normal       0.78      0.73      0.76       100
      bright       0.71      0.67      0.69       100

    accuracy                           0.74       300
   macro avg       0.74      0.74      0.74       300
weighted avg       0.74      0.74      0.74       300



In [87]:
print("Train DataFrame:", len(train_df))
print("Validation DataFrame:", len(val_df))
print("Train Dataset:", len(train_dataset))
print("Validation Dataset:", len(val_dataset))

print("\nValidation distribution:")
print(val_df["label"].value_counts().sort_index())

Train DataFrame: 1200
Validation DataFrame: 300
Train Dataset: 1200
Validation Dataset: 300

Validation distribution:
label
0    100
1    100
2    100
Name: count, dtype: int64


In [88]:
import pandas as pd

df = pd.read_csv("train.csv")

print(df.head())
print(df["label"].value_counts())

                                     id  label
0  02137a86-0743-40e0-845b-6d22d1d5cc85      0
1  025d39a8-7859-4558-9bf9-bbdd475c6100      0
2  02a2a878-c5a4-490a-8061-6b2f4ac3b6d0      0
3  047f7996-9f0d-4a04-ae7f-24e246c407c7      0
4  052a9d62-a31f-4e4f-9a76-edac2d2ae95d      0
label
0    500
1    500
2    500
Name: count, dtype: int64


In [89]:
idx_to_class = {
    0: "bright",
    1: "dark",
    2: "normal"
}

In [90]:
image_id = "0b8ebc21-dde0-4c94-acc0-7522b548e7fa"

print(df[df["id"] == image_id])

                                        id  label
1065  0b8ebc21-dde0-4c94-acc0-7522b548e7fa      2
